# VGG-11 BaCP at learning rate 0.1 --- the measurement that was never taken

`FAMILIES` runs VGG-11 BaCP at **0.05**, and the justification comment in `nb_common.py` says so plainly: the probe was *"measured on vgg19 magnitude 0.95"*, and VGG-11 took the same value because it *"shares the architecture and the symptom"*. That is an inference. VGG-11 has never been measured at both learning rates under one protocol.

It matters because VGG-11 is the only model with a negative mean delta in the matched sweep (-0.27), so whether that is architectural or an artefact of an untested extrapolation changes what the paper claims.

## What the existing records do and do not say

| VGG-11 magnitude | archive: LR 0.1, 60+50, no split | sweep: LR 0.05, 50+25, 9:1 |
|---|---|---|
| dense | 89.65 | 90.27 |
| 0.95 | 89.48 | 89.45 |
| 0.97 | 88.90 | 89.47 |
| 0.99 | 88.40 | 89.36 |
| 0.999 | 85.57 | 83.52 |

Those columns differ in four things at once, so they settle nothing. Note also that **no VGG-11 I.P. record exists in the archive** --- the earlier "gains" were against dense, not against a matched baseline, and at 0.1 BaCP was already below its own dense (89.48 vs 89.65).

A competing explanation for the negative delta is that the I.P. arm gained BaCP's 25-epoch fine-tune for matched compute (commit `2f991ce`), lifting it to 90.21 --- above anything VGG-11 BaCP has scored at either learning rate. This notebook does not test that; it tests only the learning rate.

Records are written under a `lr0.1` **variant** suffix, so they sit beside the sweep's 0.05 records instead of satisfying their keys.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Sanity check

Halts before any GPU time if the arm is not what it claims.

In [ ]:
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
SEED, GPU  = 1, 0

# Sequential, so the full 24 workers -- the concurrency experiment that
# motivated 6 is recorded above and came out negative.
cells = [nb.make_cell('vgg11', 'bacp', seed=SEED, pruner='magnitude',
                      sparsity=s, variant='lr0.1',
                      learning_rate=0.1, num_workers=24)
         for s in SPARSITIES]

for c in cells:
    cfg = c['config']
    assert cfg['learning_rate'] == 0.1,  f"lr is {cfg['learning_rate']}, want 0.1"
    assert cfg['num_workers'] == 24,     f"workers {cfg['num_workers']}, want 24"
    assert c['key'].endswith('.lr0.1'),  f"key {c['key']} lacks the variant suffix"
    # everything else must match the sweep's BaCP arm exactly
    ref = nb.FAMILIES['vgg11']['bacp']
    for k in ('epochs', 'epochs_ft', 'delta_T', 'val_split', 'tau',
              'contrastive_mode', 'proj_mode', 'optimizer_type_ft'):
        assert cfg[k] == ref[k], f'{k}: {cfg[k]} != sweep {ref[k]}'
    print(c['key'])

print(f'\n{len(cells)} cells; the ONLY difference from the sweep arm is '
      f'learning_rate 0.05 -> 0.1 (and loader workers, which cannot affect accuracy).')
assert nb.sanity_check(cells), 'sanity check failed'

## Run --- sequentially

`run_parallel` was tried here first and **measured slower**. Four cells reached
~20.25 of 50 contrastive epochs in 33 min (1.63 min/epoch/cell) against the
sweep's sequential rate of ~0.354 min/contrastive-epoch: a 4.6x per-cell
slowdown against an ideal 4.0x, so **0.87x aggregate throughput**. A single
cell already saturates the A100; the 4.15 GB peak memory was never evidence
that it did not. Running one at a time.

Expect roughly 95 min for the four cells (sweep-measured VGG-11 BaCP magnitude:
28.5 / 26.3 / 21.1 / 17.7 min).

In [ ]:
nb.run_group(cells, gpu=GPU)

## Verdict

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k:
        acc[k] = r.get('test_acc_pct')

print(f'{"sparsity":>9}  {"I.P.":>7} {"BaCP .05":>9} {"BaCP .10":>9}   '
      f'{"d(.05)":>7} {"d(.10)":>7}')
better = 0
for s in SPARSITIES:
    ip  = acc.get(f'static.prune.vgg11.cifar10.s{s}.magnitude.seed{SEED}')
    b05 = acc.get(f'static.bacp.vgg11.cifar10.s{s}.magnitude.seed{SEED}')
    b10 = acc.get(f'static.bacp.vgg11.cifar10.s{s}.magnitude.seed{SEED}.lr0.1')
    f = lambda v: f'{v:7.2f}' if v is not None else '      -'
    d = lambda a, b: f'{a-b:+7.2f}' if (a is not None and b is not None) else '      -'
    print(f'{s:>9}  {f(ip)} {f(b05)} {f(b10)}   {d(b05,ip)} {d(b10,ip)}')
    if b10 is not None and b05 is not None and b10 > b05:
        better += 1

print(f'\n0.1 beat 0.05 in {better}/{len(SPARSITIES)} cells.')
print('If 0.1 wins under this matched protocol, FAMILIES should revert vgg11 to')
print('0.1 and the justification comment in nb_common.py is wrong. If it does')
print('not, VGG-11 negative delta is architectural and belongs in the paper as')
print('the boundary condition: no damage to repair means nothing to recover.')